In [40]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn



In [41]:
# Inpsect the dataset

data = np.load("processed_dataset.npz", allow_pickle=True)

print(data.files)

for key in data.files: # data.files is like dictionary
    print(key, data[key].shape)

features_columns = ['Accel_X', 'Accel_Y', 'Accel_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z']
label_columns = ['Label']

df1 = pd.DataFrame(data['x_train'][0], columns=features_columns)

print(data['y_train'][0])
df1



['x_train', 'y_train', 'x_test', 'y_test']
x_train (15640, 40, 6)
y_train (15640,)
x_test (3911, 40, 6)
y_test (3911,)
T


,Accel_X,Accel_Y,Accel_Z,Gyro_X,Gyro_Y,Gyro_Z
0,5.336676,-3.505111,10.766725,-0.083536,0.521467,0.078873
1,3.045424,-2.487576,9.713276,-0.19172,-0.181994,0.040769
2,4.357447,-2.8491,7.450756,-0.135096,-0.677348,-0.070346
3,3.608062,-3.315969,8.207323,-0.024914,0.125904,-0.112181
4,3.89776,-3.10528,8.418014,0.083936,-0.432069,-0.135363
5,5.549759,-3.205836,9.995792,0.045698,0.084469,-0.220231
6,3.861847,-3.366248,9.057265,-0.066216,-0.018652,-0.10392
7,4.168305,-3.229778,8.8777,-0.088332,-0.204643,-0.133498
8,4.019865,-3.29921,9.129091,-0.072345,-0.213969,-0.088199
9,4.795586,-3.275268,8.576031,-0.013989,-0.166539,-0.111648


In [42]:
X_train = data['x_train'].astype(np.float32) # converting data to float for tensors
X_test = data['x_test'].astype(np.float32)

encoder = LabelEncoder()

y_train = encoder.fit_transform(data['y_train']) # training data also teaching mapping
y_test = encoder.transform(data['y_test'])

print(encoder.classes_)



['C' 'D' 'J' 'T' 'U' 'Wa' 'Wr']


In [43]:
X_train = torch.tensor(X_train, dtype=torch.float32) # float because neural networks performs floating point maths [Redundant because done already]
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long) # long because CrossEntropyLoss expects integer class indices
y_test = torch.tensor(y_test, dtype=torch.long)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

torch.Size([15640, 40, 6])
torch.Size([3911, 40, 6])
torch.Size([15640])
torch.Size([3911])


In [44]:
train_dataset = TensorDataset(X_train, y_train)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    drop_last=False
)

In [45]:
class HARnet(nn.Module):
    def __init__(self):
        super().__init__()

        self.flatten = nn.Flatten()  # converts (64, 40, 6) to (64 (batch size), 240 (input features)) which neural network expects

        self.fc1 = nn.Linear(40*6, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, len(encoder.classes_))

        
    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = HARnet()
print(model)

HARnet(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=240, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=7, bias=True)
)


In [46]:
criterion = nn.CrossEntropyLoss()

# Convert logits into probabilities (using Softmax internally).
# Compare those probabilities with the true label.
# Produce a single number called the loss.

optimizer = torch.optim.Adam(
    model.parameters(), # all weights and biases
    lr = 0.001
)

In [47]:
num_epochs = 10

for epoch in range(num_epochs):

    model.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        print(
        f"Epoch [{epoch+1}/{num_epochs}], "
        f"Loss: {running_loss/len(train_loader):.4f}"
    )

Epoch [1/10], Loss: 0.0089
Epoch [1/10], Loss: 0.0150
Epoch [1/10], Loss: 0.0205
Epoch [1/10], Loss: 0.0246
Epoch [1/10], Loss: 0.0284
Epoch [1/10], Loss: 0.0324
Epoch [1/10], Loss: 0.0363
Epoch [1/10], Loss: 0.0389
Epoch [1/10], Loss: 0.0416
Epoch [1/10], Loss: 0.0462
Epoch [1/10], Loss: 0.0495
Epoch [1/10], Loss: 0.0517
Epoch [1/10], Loss: 0.0540
Epoch [1/10], Loss: 0.0568
Epoch [1/10], Loss: 0.0608
Epoch [1/10], Loss: 0.0629
Epoch [1/10], Loss: 0.0657
Epoch [1/10], Loss: 0.0685
Epoch [1/10], Loss: 0.0714
Epoch [1/10], Loss: 0.0743
Epoch [1/10], Loss: 0.0762
Epoch [1/10], Loss: 0.0784
Epoch [1/10], Loss: 0.0809
Epoch [1/10], Loss: 0.0826
Epoch [1/10], Loss: 0.0844
Epoch [1/10], Loss: 0.0868
Epoch [1/10], Loss: 0.0888
Epoch [1/10], Loss: 0.0909
Epoch [1/10], Loss: 0.0932
Epoch [1/10], Loss: 0.0956
Epoch [1/10], Loss: 0.0976
Epoch [1/10], Loss: 0.0991
Epoch [1/10], Loss: 0.1011
Epoch [1/10], Loss: 0.1034
Epoch [1/10], Loss: 0.1057
Epoch [1/10], Loss: 0.1090
Epoch [1/10], Loss: 0.1110
E

In [48]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    outputs = model(X_test)

    _, predicted = torch.max(outputs, dim=1)

    total += y_test.size(0)

    correct += (predicted == y_test).sum().item()

accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 89.31%
